In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
from muon import prot as pt
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
import io

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib import colors
from matplotlib_venn import venn2
import matplotlib.image as mpimg

#Conversions
import biomart

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

#Global figure options
plt.rcParams["figure.figsize"] = (3.5, 3)
plt.rcParams["font.family"] = "Arial"
sb.set_theme(style="ticks", font="Arial")

# Helper functions from Mimi

In [ ]:
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRAC', 'TRB', 'TRDV', 'TRDC', 'TRG']

def filter_tcr_genes(adata):
    non_tcr_genes = adata.var_names
    for prefix in tcr_gene_prefixs:
        non_tcr_genes = [el for el in non_tcr_genes if not el.startswith(prefix)]
    adata = adata[:, non_tcr_genes]
    return adata

def filter_high_var(adata, n):
    #sc.pp.highly_variable_genes(
    #adata,
    #flavor="seurat_v3",
    #n_top_genes=n,
    #layer="counts",
    #batch_key="pool",
    #subset=True,
    #span = 0.75) #default = 0.3. It's the fraction of cells used for estimation of hvg - if I have 150 cells in one sample but 150.000 cells in total then this sample becomes an issue. 
    
    sc.pp.highly_variable_genes(adata, n_top_genes=n)
    adata = adata[:, adata.var['highly_variable']].copy()
    return adata


def calculate_umap(adata, n_high_var=5000, remove_tcr_genes=True):
    """
    Calculates UMAP coordinates
    :param adata: AnnData object
    :param n_high_var: how many variable genes to use, None uses raw data
    :param remove_tcr_genes: whether to remove TCR genes
    :return: results stored in adata object
    """
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)            
    sc.tl.umap(adata_tmp)
    adata.obsm['X_umap'] = adata_tmp.obsm['X_umap']


def calculate_leiden(adata, resolution, n_high_var=5000, remove_tcr_genes=True):
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)
    sc.tl.leiden(adata_tmp, resolution=resolution)

    leiden_column_name = f"leiden_{resolution}"
    adata.obs[leiden_column_name] = adata_tmp.obs['leiden']

# Loading data
See the following for a discussion on data concatenation in muon: https://github.com/scverse/muon/issues/127

In [ ]:
sdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_main_dataset_clustered.h5mu")


In [ ]:
sdata

# Basic annotation - L1_ann (T cells versus others)

In [ ]:
plt.rcParams["figure.figsize"] = (7, 6)
sc.pl.umap(sdata["gex"], color=["leiden_4.5", "chain_pairing"],  legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=1)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
Myeloid_markers2 = {
    "T-cell status": ["CD3E", "TRAC"],
    "APC status": ["CD74", "HLA-DRA", "HLA-DRB1"],
    "DC": ["FLT3"], 
    "cDC1": ["CLEC9A", "XCR1",  "IDO1"], 
    "cDC2": ["CLEC9A","CLEC10A", "CD1C", "FLT3"],
    "Monocyte": ["FCN1", "S100A8", "VCAN", "CSF3R", "CD14", "FCGR3A"],
    "BAM": ["F13A1", "LYVE1","CD163", "STAB1" ,"MRC1"],    # "BAM": ["C1QA", "TREM2", "F13A1","LYVE1", "CD163", "MRC1"],
    "Microglia": ["APOE", "TREM2", "SORL1","CX3CR1", "SLC2A5",  "SPP1", "P2RY12", "TMEM119", "SERPINE2" ,"SALL1" ], # "Microglia": ["APOE", "SORL1", "SLC2A5", "BHLHE41", "SPP1", "P2RY12"], 
    "B cells": [ "CD79A", "MS4A1", "CD19", "TNFRSF13B", "IGHG1", "IGHG2", "IGHG3"], #"IGKC", "IGHM", "IGHD", "IGHG1", "IGHG2", "IGHG3"], 
    "Plasma cell": ["PRDM1", "XBP1", "CD38",  "SLAMF7", "MZB1"],
    "pDC" :["IRF8","JCHAIN", "PTCRA", "SMIM5", "LAMP5"],
    "NK cell": ["NCAM1", "NCR1", "KIR2DL4"], 
    "MAIT cell" : ["TRAV1-2", "SLC4A10"], 
    "Ribosomal_genes": ["RPLP1", "EEF1A1", "RPL28"], 
}

sc.pl.dotplot(sdata["gex"], Myeloid_markers2, groupby="leiden_4.5", standard_scale="var", color_map='RdYlBu_r')


In [ ]:
###### plt.rcParams["figure.figsize"] = (7, 6)
APC_Doub = ["58", "54", "62", "61", "41", "38", "50", "22", "47","49", "34"]
sc.pl.umap(sdata["gex"], color=["leiden_4.5"],  groups=APC_Doub, legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=5)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Signature-based annotation

In [ ]:
# #BAM - from Ostkampf, who compiled it from Van Hove et al 2018
# ## Collected from figure 2 of the Van Hove manuscript
# VanHove_BAM = ["APOE", "MS4A7", "MS4A6A", "LYZ", "CLEC4A", "IFITM2", "TGFBI", "CYBB", "MNDA", "IFI27L2", "PF4", "MSR1", "AOAH", "DAB2"]
# sc.tl.score_genes(sdata["gex"], 
#                       VanHove_BAM,  
#                       score_name='VanHove_BAM')


# plt.rcParams["figure.figsize"] = (7, 6)
# sc.pl.umap(sdata["gex"], color=["VanHove_BAM"],   legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=5, color_map='RdYlBu_r')
# plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
# #Microglial cells - from Ostkampf
# VanHove_MG = genes = [
#     "SALL1", "ADGRG1", "SLC2A5", "IL1A", "GOLM1", "ADAMTS1", "SELPLG", "SPARC",
#     "GEM", "SERPINE2", "P2RY12", "FSCN1", "RTN4RL1", "ATP6V0A2", "ARHGAP5",
#     "SIGLEC6", "CD33", "SMAD7", "P2RY13", "TMEM119", "FAM102B", "SMAP2",
#     "CSF3R", "SOGA1", "KCNK6", "WSB1", "RGS1", "LDHB", "GPR34", "GLUL",
#     "CST3", "LHFPL2", "GPR155", "RASAL3", "DTX4", "SRGAP2C", "SRGAP2B",
#     "FRMD4A", "SPATA13", "QKI", "ENTPD1", "HEXB", "TANC2", "BIN1", "TREM2",
#     "OLFML3", "CTSF", "ELMO1", "ASPH", "GNA15", "CMTM6", "HERPUD1", "BRD2",
#     "PDE3B", "TGFBR1", "DAGLB", "NFKBIA", "BASP1", "ARSB", "INPP5D", "FMNL3",
#     "SLC29A3", "SIPA1", "CX3CR1", "SNX18", "F11R", "ITGB5", "FCRL2"
# ]
# sc.tl.score_genes(sdata["gex"], 
#                       VanHove_MG,  
#                       score_name='VanHove_MG')

# plt.rcParams["figure.figsize"] = (7, 6)
# sc.pl.umap(sdata["gex"], color=["VanHove_MG"],   legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=5, color_map='RdYlBu_r')
# plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
# ## Sankowski - CAMs - log2FC > 0.5 ´postnatal
# CAM_sankowski = ["MRC1","RNASE1","F13A1","NRP1","SELENOP","XIST","PDGFC","MYO5A","WWP1","PID1","CD163","RNF150","COLEC12","DPYD","THRB","TRPS1","SLC9A9","MS4A7","EPS15","FRMD4B","VAV3","C1QA","RGL1","IQGAP2","ASPH","ARHGAP18","MCTP1","WDFY3","EDA","ZEB2","TGFBI","FCHO2","MS4A6A","LYVE1","STARD13","ME1","TTN","STAB1","WLS","SPRED1","THBS1","MBNL1","PAPSS2","NFIA","SGMS1","SLCO2B1","RBPJ","FMN1","FCHSD2","ITSN1","ITGA9","FKBP5","SNX6","MS4A4A","FOXP1","TBC1D5","UTRN","AGFG1","DISP1","FGF13","PTPRM","DLEU2","CNTLN","TXNIP","EIF4G3","PER3","TPST1","MTSS1","ESR1","SLC38A6","PPP3CA","BAZ2B","TENT5A","DAPK1","TCF12","MIR99AHG","PHF21A","P2RY14","ELMO1","MS4A4E","STON2","EPB41L3","NCOA1","LDLR","IL12RB2","HELZ","EMB","ARL15","ST8SIA4","GPR155","SLC40A1","EPS8","ANKRD12","DOCK7","MAMDC2","AHR","MVB12B","FLI1","KLF7","SESTD1"]

# sc.tl.score_genes(sdata["gex"], 
#                       CAM_sankowski,  
#                       score_name='Sankowski_CAM')


# plt.rcParams["figure.figsize"] = (7, 6)
# sc.pl.umap(sdata["gex"], color=["Sankowski_CAM"],   legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=5, color_map='RdYlBu_r')
# plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
# ## Sankowski - MG - log2FC > 0.5 + post natal  cells
# MG_sankowski = ["SLC1A3","NAV3","P2RY12","HSPA1B","IPCEF1","OLR1","C3","SYNDIG1","ST6GAL1","ST6GALNAC3","GLDN","APBB1IP","GRID2","DSCAM","HSPA1A","SOCS6","LNCAROD","RTTN","BHLHE41","SORL1","HSPA6","OXR1","EPB41L2","FGD4","SERPINE1","DNAJB1","APOC2","KCNQ3","CX3CR1","PDGFB","MCF2L2","CD74","PLCG2","LHFPL2","KCNMA1","HS3ST4","B3GNT5","RASGEF1C","PLXDC2","CCL4","ARHGAP24","SDK1","SHTN1","ABCC4","MYO1E","ADAM28","BAG3","LPCAT2","PABPC1","OLFML3","CD83","ARHGAP22","ATP2C1","A2M","PRKCH","HDAC9","FRMD4A","FOXP2","TMEM156","FCGR3A","ITGAX","CCL4L2","IFNGR1","KCNIP1","TLN2","HLA-DRA","SRGAP2","FLT1","DOCK10","PTPRJ","CACNA1A","NFKB1","AKT3","IL6ST","MICOS10","ALCAM","CSGALNACT1"]

# sc.tl.score_genes(sdata["gex"], 
#                       MG_sankowski,  
#                       score_name='Sankowski_MG')


# plt.rcParams["figure.figsize"] = (7, 6)
# sc.pl.umap(sdata["gex"], color=["Sankowski_MG"],   legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=5, color_map='RdYlBu_r')
# plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
# ## Keren-Shaul DAMs
# ### Taken from Sankowski 
# sankowski_shaul_dams = ["LPL", "ITGAX", "SPP1", "APOE", "CSF1", "CLEC7A", "IGF1", "AXL", "ANKH", "CD63", "TYROBP", "CTSB", "CTSD", "CD9", "CTSZ", "CCL23", "CCL15", "TREM2"]

# sc.tl.score_genes(sdata["gex"], 
#                       sankowski_shaul_dams,  
#                       score_name='Keren-Shaul_DAM')

# plt.rcParams["figure.figsize"] = (7, 6)
# sc.pl.umap(sdata["gex"], color=["Keren-Shaul_DAM"],   legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=5, color_map='RdYlBu_r')
# plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# ##  Keren-Shaul DAMs (microglial cluster 3 vs 1) - selected on all significant genes above UMI count 2
# ## It looks as though Sankowski took stuff out?
# #  genes = ["Apoe","Ctsd","Ctss","B2m","Gm10925","Ctsb","Tyrobp","Cst7","Fth1","Gnas","Ctsl","Cd9","Grn","mmu-mir-703","Ctsz","Gm11410","Trem2","Rplp1","Lyz2","AC151602.1","Mpeg1","Hexa"]


### Annotate cell types

Sources to cite: 
- https://www.science.org/doi/10.1126/sciimmunol.abf7777  (FCN1, CCR2, and S100A8 defines monocyte vs monocyte-derived macro vs resident macro)
- https://apps.allenimmunology.org/aifi/resources/imm-health-atlas/cell-type-descriptions/monocytes/    (CTSS and FCN1 defines monocytes)
- https://apps.allenimmunology.org/aifi/resources/imm-health-atlas/cell-type-descriptions/monocytes/    (VCAN and S100A8 are particular to CD14+ monocytes)
- https://apps.allenimmunology.org/aifi/resources/imm-health-atlas/cell-type-descriptions/dendritic-cells/   DCs
- PLXDC2 is specifically expressed on macrophages, not NK, T cells or neutrophils (see Fig 5C). https://www.cell.com/developmental-cell/fulltext/S1534-5807(24)00486-6#fig5

In [ ]:
## Simple clusters to identify T cell versus myeloid cell
marker_genes = {
    "T cells": ["CD3E", "TRBC2", "THEMIS", "TRAC", "TRBC1"],
    "Antigen-presentation": ["HLA-DRA", "HLA-DRB1", "CD74"],
    "Complement genes": ["C1QA", "C1QB", "C1QC"],
    "Monocyte genes": ["FCN1","CTSS",  "CCR2", ], 
    "CD14+ Mono":["S100A8", "VCAN", "CD14"],
    "CD16+ Mono":["CD68", "FCGR3A"],
    "BAM": ["CYBB", "F13A1", "MS4A7","LYVE1", "CD163", "MRC1", "PLXDC2"],
    "BAM-associated": ["APOE", "TGFBI", "APOC1", "ITGAX"],
    "Microglia": ["SPP1", "TREM2", "CCL2", "CLEC7A", "ITGAX", "CX3CR1", "P2RY12", "TMEM119"],
    "mDCs": ["FLT3", "CLEC9A", "CD1C"],  
    "B cells": ["CD19", "MS4A1", "CD79A", "IGHG1","IGKC", "JCHAIN"],
    "pDCs": ["LILRA4", "IRF8"],
    "CITEseq": ["CD4_adt", "CD8_adt"],
    "CD4 vs CD8": ["CD4", "CD8A"],
    "MAIT" : ["TRAV1-2", "SLC4A10"],
    "GD T cells": ["TRGV9", "TRDV1"],
    "Effector": ["CCL5", "GZMK", "GZMB", "GNLY", "NKG7", "LAG3", "TIGIT"],
    "IFN-sig": ["IFI44", "IFI44L", "ISG15", "OAS2"], 
    "T naive": ["LEF1", "CCR7", "TCF7", "SELL"],
    "T reg": ["FOXP3", "IL2RA", "CTLA4"],
    "NK cells": ["NCAM1", "NCR1", "KIR2DL4", 
                 "KLRB1", #CD161
                 "KLRC2", #NKG2A specifically
                 "KLRK1", #NKG2D specifically
                 "KLRC1", #NKG2A,D,C
                 "KLRD1"], #CD94
    "Ribosomal_genes": ["RPLP1", "EEF1A1", "RPL28"]
    # "AXL+SIGLEC6+ DCs":["PPP1R14A", "AXL", "SIGLEC6"],
}

sc.pl.dotplot(sdata["gex"], marker_genes, groupby="leiden_4.5", standard_scale="var", color_map='RdYlBu_r')

In [ ]:
# ## Simple clusters to identify T cell versus myeloid cell
# marker_genes = {
#     "T cells": ["CD3E", "TRAC"],
#     "MAIT" : ["TRAV1-2", "SLC4A10"],
#     # "GD T cells": ["TRGV7", "TRGV8", "TRGV9", "TRDV1"],
#     # "NK cells": ["NCAM1", "NCR1", "KIR2DL4", 
#     #              "KLRB1", #CD161
#     #              "KLRC2", #NKG2A specifically
#     #              "KLRK1", #NKG2D specifically
#     #              "KLRC1", #NKG2A,D,C
#     #              "KLRD1"], #CD94
#     "Antigen-presentation": ["HLA-DRA", "HLA-DRB1", "CD74"],
#     "Complement genes": ["C1QA", "C1QB", "C1QC"],
#     "CD14+ Mono":["CD14"],
#     "CD16+ Mono":["CD68", "FCGR3A"],
#     "BAM": ["CYBB", "F13A1", "MS4A7","LYVE1", "CD163", "MRC1"],
#     "BAM-associated": ["APOE", "TGFBI", "APOC1", "ITGAX"],
#     "Microglia": ["SPP1", "TREM2", "CCL2", "CLEC7A", "ITGAX", "CX3CR1", "P2RY12", "TMEM119"],
#     "mDCs": ["FLT3", "CLEC9A", "CD1C", "CD1E", "FCER1A"],
#         "pDCs": ["LILRA4", "IRF8"],
#     "B cells": ["CD19", "MS4A1", "CD79A", "IGHG1","IGKC", "JCHAIN"],
# }

# sc.pl.dotplot(sdata["gex"], marker_genes, groupby="leiden_4.5", standard_scale="var", color_map='RdYlBu_r')

In [ ]:
annotation_dict = {
    '0': 'T cells',
    '1': 'T cells',
    '2': 'T cells',
    '3': 'T cells',
    '4': 'T cells',
    '5': 'T cells',
    '6': 'T cells',
    '7': 'T cells',
    '8': 'T cells',
    '9': 'MAIT cells',
    '10': 'T cells',
    '11': 'T cells',
    '12': 'T cells',
    '13': 'T cells',
    '14': 'T cells',
    '15': 'T cells',
    '16': 'T cells',
    '17': 'T cells',
    '18': 'T cells',
    '19': 'T cells',
    '20': 'T cells',
    '21': 'T cells',
    '22': 'MG/BAMs',
    '23': 'T cells',
    '24': 'T cells',
    '25': 'T cells',
    '26': 'T cells',
    '27': 'T cells',
    '28': 'T cells',
    '29': 'T cells',
    '30': 'T cells',
    '31': 'T cells',
    '32': 'T cells',
    '33': 'T cells',
    '34': 'Multiplets',
    '35': 'T cells',
    '36': 'T cells',
    '37': 'T cells',
    '38': 'DCs',
    '39': 'T cells',
    '40': 'MAIT cells',
    '41': 'Multiplets',
    '42': 'T cells',
    '43': 'T cells',
    '44': 'T cells',
    '45': 'T cells',
    '46': 'T cells',
    '47': 'MG/BAMs',
    '48': 'T cells',
    '49': 'MG/BAMs',
    '50': 'Monocytes',
    '51': 'T cells',
    '52': 'T cells',
    '53': 'T cells',
    '54': 'CD20+ B cells',
    '55': 'NK cells',
    '56': 'T cells',
    '57': 'T cells',
    '58': 'Multiplets',
    '59': 'T cells',
    '60': 'T cells',
    '61': 'pDCs',
    '62': 'Plasmablasts',
    '63': 'T cells',
    '64': 'T cells',
    '65': 'T cells',
}
 
# Add cell type column based on annotation
sdata["gex"].obs['L1_ann'] = [annotation_dict[sample] for sample in sdata["gex"].obs['leiden_4.5']]

In [ ]:
sc.set_figure_params(figsize=(7,6))
sc.pl.umap(sdata["gex"], color=["leiden_4.5", "L1_ann"], legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=5)

# Summary of findings

In [ ]:
sdata["gex"].obs["L1_ann"].value_counts()

In [ ]:
order = [
    "T cells",
    "MAIT cells",
    "Multiplets",
    "MG/BAMs",
    "Monocytes",
    "DCs",
    "CD20+ B cells",
    "Plasmablasts",
    "NK cells",
    "pDCs"
]

sdata["gex"].obs["L1_ann"] = pd.Categorical(
    sdata["gex"].obs["L1_ann"],
    categories=order,
    ordered=True
)

marker_genes = {
        "T cells": ["CD3E", "LCK", "TRAC"],
        "MAIT" : ["TRAV1-2", "SLC4A10"],
        "MG/BAM":  ["TREM2", "APOE", "LYVE1", "SPP1"],
        "Monocytes" : ["FCN1", "VCAN"], 
        "mDCs": ["CD1C", "FLT3", "CLEC9A"],
        "B cells": ["MS4A1", "CD19", "IGHG1"],
        "NK cells": ["NCAM1", "NCR1"], 
        "pDCs": ["IL3RA", "PTCRA", "IRF7"]}

## Visualize
dp = sc.pl.dotplot(sdata["gex"], marker_genes, groupby="L1_ann", standard_scale="var", return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5).show()

# CD4 + CD8 assignment

### Save co-receptor data in .obs. 
Count the number of cells annotated depending on different ways of annotating CD8 T cells

In [ ]:
# Get the index of the CD4 and CD8 genes
cd4_idx = np.where(sdata["gex"].var_names == "CD4")[0][0]
cd4_expr = sdata["gex"].X[:, cd4_idx]
cd4_expr = cd4_expr.toarray().flatten() if not isinstance(cd4_expr, np.ndarray) else cd4_expr.flatten()
sdata["gex"].obs["CD4_expr"] = cd4_expr

cd8_idx = np.where(sdata["gex"].var_names == "CD8A")[0][0]
cd8_expr = sdata["gex"].X[:, cd8_idx]
cd8_expr = cd8_expr.toarray().flatten() if not isinstance(cd8_expr, np.ndarray) else cd8_expr.flatten()
sdata["gex"].obs["CD8A_expr"] = cd8_expr

cd8_idx = np.where(sdata["gex"].var_names == "CD8B")[0][0]
cd8_expr = sdata["gex"].X[:, cd8_idx]
cd8_expr = cd8_expr.toarray().flatten() if not isinstance(cd8_expr, np.ndarray) else cd8_expr.flatten()
sdata["gex"].obs["CD8B_expr"] = cd8_expr

cd8_idx = np.where(sdata["prot"].var_names == "CD8_adt")[0][0]
cd8_expr = sdata["prot"].X[:, cd8_idx]
cd8_expr = cd8_expr.toarray().flatten() if not isinstance(cd8_expr, np.ndarray) else cd8_expr.flatten()
sdata["prot"].obs["CD8_prot_expr"] = cd8_expr
sdata["gex"].obs["CD8_prot_expr"]=sdata["prot"].obs["CD8_prot_expr"].copy()

## Sanity check - CSF expression, where it is more difficult to annotate

In [ ]:
ax = sc.pl.scatter(
    sdata['gex'],
    x="CD8A_expr",
    y="CD8B_expr",
    color="Compartment",
    groups="CSF",
    size=10,
    show=False
)

ax.axhline(y=0.1, color='black', linestyle='--', lw=0.5)
ax.axvline(x=0.1, color='black', linestyle='--', lw=0.5)
sb.despine()
ax.grid(False)

plt.show()

## Sanity check - number of CD4 and CD8 T cells identified by GEX

In [ ]:
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[(obs_df["Compartment"] == "PB") & (obs_df["L1_ann"].isin(["T cells", "MAIT cells"]))]
obs_df = obs_df[obs_df["cd4cd8_cs_assignment"] == "CD8"]
obs_df = obs_df[obs_df["chain_pairing"] != "two full chains"]
obs_df["CD8B_cat"] = np.where(obs_df["CD8B_expr"] > 0, True, False)
obs_df[["L1_ann", "CD8B_cat"]].value_counts()

## Interpretation - CD8B sucessfully detected in 554746 CD8 T cells out of 55204 + 9207 T cells. 


In [ ]:
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[(obs_df["Compartment"] == "PB") & (obs_df["L1_ann"].isin(["T cells", "MAIT cells"]))]
obs_df = obs_df[obs_df["cd4cd8_cs_assignment"] == "CD8"]
obs_df = obs_df[obs_df["chain_pairing"] != "two full chains"]
obs_df["CD8_cat"] = np.where((obs_df["CD8B_expr"] > 0) & (obs_df["CD8A_expr"] > 0), True, False)
obs_df[["L1_ann", "CD8_cat"]].value_counts()

## Interpretation - CD8B sucessfully detected in 554746 CD8 T cells out of 55204 + 9207 T cells. 


# Step 1: Assignment via gene expression
CD8A and CD8B seems equal, but I will go with CD8B for the following reasons:
- CD8B was on a separate project necessary to use, because we found CD4+CD8A+ T-cell clonotypes
- CD8B is slightly higher expressed judged by a scatter plot in CSF compared to CD8A
- CD8B finds slightly my true positive CD8 T cells in peripheral blood (reference annotation made by CITEseq)

In [ ]:
# Create the violin plot (set show=False to avoid automatic plotting)
sc.pl.violin(
    sdata["gex"],             # Your AnnData object
    keys=["CD4", "CD8B"],  # Genes to plot
    groupby=None,             # No grouping; overall distribution
    rotation=90,              # Rotate x-axis labels
    stripplot=False,          # Remove the individual dots
    jitter=False,             # Turn off jittering
    show=False                # Prevent automatic plotting
)

# Get the current axis and add a horizontal line at y=20
ax = plt.gca()  # Get current axis
ax.axhline(y=0.1, color='black', linestyle='--')  # Black dashed line at y=20

# Show the plot
plt.show()

## Define cd4cd8_gex_assignment 

In [ ]:
# Assign categories for each cell
sdata["gex"].obs["cd4cd8_gex_assignment"] = pd.Categorical(
    np.where((sdata["gex"].obs["CD8B_expr"] == 0) & (sdata["gex"].obs["CD8A_expr"] == 0) & (sdata["gex"].obs["CD4_expr"] > 0), "CD4",
             np.where(((sdata["gex"].obs["CD8B_expr"] > 0) | (sdata["gex"].obs["CD8A_expr"] > 0)) & (sdata["gex"].obs["CD4_expr"] > 0), "CD4/CD8", 
                     np.where((sdata["gex"].obs["CD8A_expr"] > 0) | (sdata["gex"].obs["CD8B_expr"] > 0), "CD8", "negative"))))

## Sanity check - cell numbers, CD4_adt, CD8_adt expression, double positive T cells, and how "cleanly" annotated cells expressed CD4 and CD8A/CD8B

In [ ]:
sdata["gex"].obs["cd4cd8_gex_assignment"].value_counts()

In [ ]:
sdata["prot"].obs["cd4cd8_gex_assignment"] = sdata["gex"].obs["cd4cd8_gex_assignment"].copy()

In [ ]:
sdata['prot'].obs["L1_ann"] = sdata['gex'].obs["L1_ann"].copy()

ax = sc.pl.scatter(
    sdata['prot'],
    x="CD4_adt",
    y="CD8_adt",
    groups="CD4/CD8",
    color="cd4cd8_gex_assignment",
    show=False,
    size=20
)

ax.axhline(y=9, color='black', linestyle='--', lw=0.5)
ax.axvline(x=9, color='black', linestyle='--', lw=0.5)

plt.show()

In [ ]:
coreceptors = {
    "T-cell status": ["CD3E", "CD4", "CD8A", "CD8B"]
}

sc.pl.dotplot(sdata["gex"], coreceptors, groupby="cd4cd8_gex_assignment", standard_scale="var", color_map='RdYlBu_r')


## Visualize _gex_ and _cs_ assignment

In [ ]:
# Ensure category order is ['low', 'high'] so 'grey' maps to 'low' and 'black' to 'high'
sdata["gex"].obs["cd4cd8_gex_assignment"] = sdata["gex"].obs["cd4cd8_gex_assignment"].cat.reorder_categories(["CD4", "CD8", "CD4/CD8", "negative"])
plt.rcParams["figure.figsize"] = (3.5, 3)
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_gex_assignment"],
    palette=["blue", "orange", "red", "lightgrey"],  # First color corresponds to first category
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

# CD4 assignment
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_gex_assignment"],
    groups=["CD4"],
    palette=["blue", "orange", "red", "lightgrey"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

# CD8 assignment
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_gex_assignment"],
    groups=["CD8"],
    palette=["blue", "orange", "red", "lightgrey"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

# CD4/CD8 assignment
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_gex_assignment"],
    groups=["CD4/CD8"],
    palette=["blue", "orange", "red", "lightgrey"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)


# Explicit palette mapping
palette2 = {
    "negative": "black",
    "CD4/CD8": "lightgrey",
    "CD4": "lightgrey",
    "CD8": "lightgrey"
}

sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_gex_assignment"],
    groups=["negative"],
    palette=palette2,
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Step 2: Assignment via surface expression (already done in preprocessing of individual files)

### Fix annotation labels

In [ ]:
# Ensure 'cd4cd8_cs_assignment' column is categorical
sdata["gex"].obs["cd4cd8_cs_assignment"] = sdata["prot"].obs["cd4cd8_cs_assignment"].astype("category")

# Rename slightly
sdata["gex"].obs["cd4cd8_cs_assignment"] = pd.Categorical(np.where(sdata["gex"].obs["cd4cd8_cs_assignment"] == "CD8", "CD8_adt", 
                                                                   np.where(sdata["gex"].obs["cd4cd8_cs_assignment"] == "CD4", "CD4_adt", 
                                                                            np.where(sdata["gex"].obs["cd4cd8_cs_assignment"] == "CD4/CD8", "CD4_adt/CD8_adt", 
                                                                                     np.where(sdata["gex"].obs["cd4cd8_cs_assignment"] == "adt_negative", "adt_negative", "error")))))

In [ ]:
# Define valid categories including 'adt_negative'
valid_categories = ["CD4_adt", "CD8_adt", "CD4_adt/CD8_adt", "adt_negative"]

# Ensure 'cd4cd8_cs_assignment' column is categorical and set the valid categories first
sdata["gex"].obs["cd4cd8_cs_assignment"] = sdata["gex"].obs["cd4cd8_cs_assignment"].astype("category")

# Set the categories explicitly, including 'adt_negative'
sdata["gex"].obs["cd4cd8_cs_assignment"] = sdata["gex"].obs["cd4cd8_cs_assignment"].cat.set_categories(valid_categories)

# Replace NaN values in 'cd4cd8_cs_assignment' with 'adt_negative'
sdata["gex"].obs["cd4cd8_cs_assignment"].fillna("adt_negative", inplace=True)


In [ ]:
# Ensure category order is ['low', 'high'] so 'grey' maps to 'low' and 'black' to 'high'


# sdata["gex"].obs["cd4cd8_cs_assignment"] = sdata["gex"].obs["cd4cd8_cs_assignment"].cat.reorder_categories(["CD4_adt", "CD8_adt", "CD4_adt/CD8_adt", "adt_negative"])
plt.rcParams["figure.figsize"] = (3.5, 3)
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_cs_assignment"],
    palette=["blue", "orange", "red", "lightgrey"],  # First color corresponds to first category
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

# CD4 assignment
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_cs_assignment"],
    groups=["CD4_adt"],
    palette=["blue", "orange", "red", "lightgrey"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

# CD8 assignment
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_cs_assignment"],
    groups=["CD8_adt"],
    palette=["blue", "orange", "red", "lightgrey"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

# CD4/CD8 assignment
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_cs_assignment"],
    groups=["CD4_adt/CD8_adt"],
    palette=["blue", "orange", "red", "lightgrey"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

# adt_negative
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_cs_assignment"],
    groups=["adt_negative"],
    palette=["blue", "orange", "red", "black"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Step 3: Combined assignment using gex and correcting negatives

In [ ]:
sdata["gex"].obs[["cd4cd8_cs_assignment", "cd4cd8_gex_assignment"]]

# Create confusion matrix
confusion_matrix = pd.crosstab(sdata["gex"].obs['cd4cd8_cs_assignment'], sdata["gex"].obs['cd4cd8_gex_assignment'], rownames=['Prot'], colnames=['Gex'])

print(confusion_matrix)

In [ ]:
# VERSION 2
# ASSIGN NEGATIVE DROPOUTS BASED ON SURFACE EXPRESSION FIRST
# If CS is CD4_adt, then CD4. 
# If CS is CD8_adt, then CD8. 
# If CS is CD4_adt/CD8_adt,  then CD4/CD8
# If CS is negative, use GEX information. Default to negative

# CD4 assignment
sdata["gex"].obs["cd4cd8_combined_assignment"] = pd.Categorical(
    np.where(sdata["gex"].obs["cd4cd8_cs_assignment"] == "CD4_adt", "CD4",  
             np.where(sdata["gex"].obs["cd4cd8_cs_assignment"] == "CD8_adt", "CD8",  
                      np.where(sdata["gex"].obs["cd4cd8_cs_assignment"] == "CD4_adt/CD8_adt", "CD4/CD8",  
                               np.where(sdata["gex"].obs["cd4cd8_cs_assignment"] == "adt_negative", 
                               np.where(sdata["gex"].obs["cd4cd8_gex_assignment"] == "CD4", "CD4",
                               np.where(sdata["gex"].obs["cd4cd8_gex_assignment"] == "CD8", "CD8",
                               np.where(sdata["gex"].obs["cd4cd8_gex_assignment"] == "CD4/CD8", "CD4/CD8", "negative"))), "negative")))))


sdata["gex"].obs["cd4cd8_combined_assignment"].value_counts()

In [ ]:
sdata["gex"].obs[["cd4cd8_cs_assignment", "cd4cd8_combined_assignment"]]

# Create confusion matrix
confusion_matrix = pd.crosstab(sdata["gex"].obs['cd4cd8_cs_assignment'], sdata["gex"].obs['cd4cd8_combined_assignment'], rownames=['Prot'], colnames=['combined (gex and cs)'])

print(confusion_matrix)

In [ ]:
# Create confusion matrix
confusion_matrix = pd.crosstab(sdata["gex"].obs['cd4cd8_gex_assignment'], sdata["gex"].obs['cd4cd8_combined_assignment'], rownames=['cd4cd8_gex_assignment'], colnames=['combined (gex and cs)'])

print(confusion_matrix)

In [ ]:
# Create confusion matrix
confusion_matrix = pd.crosstab(sdata["gex"].obs['cd4cd8_gex_assignment'], sdata["gex"].obs['cd4cd8_cs_assignment'], rownames=['cd4cd8_gex_assignment'], colnames=['cd4cd8_cs_assignment'])

print(confusion_matrix)

In [ ]:
plt.rcParams["figure.figsize"] = (3.5, 3)
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_combined_assignment"],
    palette=["blue",  "red", "orange", "lightgrey"],  # First color corresponds to first category
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_combined_assignment"],
    groups=["CD4"],
    palette=["blue",  "red", "orange", "lightgrey"],  # First color corresponds to first category
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_combined_assignment"],
    groups=["CD8"],
    palette=["blue",  "red", "orange", "lightgrey"],  # First color corresponds to first category
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_combined_assignment"],
    groups=["CD4/CD8"],
    palette=["blue",  "red", "orange", "lightgrey"],  # First color corresponds to first category
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_combined_assignment"],
    groups=["negative"],
    palette=["lightgrey", "black"],  # First color corresponds to first category
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
sdata["gex"].obs[["Compartment", "cd4cd8_combined_assignment"]]

# Create confusion matrix
confusion_matrix = pd.crosstab(sdata["gex"].obs['Compartment'], sdata["gex"].obs['cd4cd8_combined_assignment'], rownames=['Compartment'], colnames=['combined (gex and cs)'])

print(confusion_matrix)

#### Visualize the trend from gex to combined for PB

In [ ]:
# Step 1: Build both confusion matrices
confusion_matrix_gex = pd.crosstab(
    sdata["gex"].obs['cd4cd8_gex_assignment'],
    sdata["gex"].obs['Compartment'],
    rownames=['cd4cd8_gex_assignment'],
    colnames=['Compartment']
)

confusion_matrix_combined = pd.crosstab(
    sdata["gex"].obs['cd4cd8_combined_assignment'],
    sdata["gex"].obs['Compartment'],
    rownames=['cd4cd8_combined_assignment'],
    colnames=['Compartment']
)

# Step 2: Ensure consistent ordering of assignment categories
assignments = ["CD4", "CD4/CD8", "CD8", "negative"]
confusion_matrix_gex = confusion_matrix_gex.reindex(assignments).fillna(0)
confusion_matrix_combined = confusion_matrix_combined.reindex(assignments).fillna(0)

# Step 3: Add suffixes to columns to differentiate gex/combined
confusion_matrix_gex.columns = [f"{col}_gex" for col in confusion_matrix_gex.columns]
confusion_matrix_combined.columns = [f"{col}_combined" for col in confusion_matrix_combined.columns]

# Step 4: Concatenate both matrices column-wise
merged_matrix = pd.concat([confusion_matrix_gex, confusion_matrix_combined], axis=1)

# Step 5: Transpose for plotting (samples on x-axis)
merged_matrix = merged_matrix.T  # rows = compartment_type, columns = assignment types

# 👉 Step 6: Sort index so CSF_x come first
merged_matrix = merged_matrix.sort_index(key=lambda x: x.str.contains("CSF"), ascending=False)

# Step 7: Plot with consistent grey-scale colors
colors = {
    "CD4": "#f4cccc",        # Weak red
    "CD4/CD8": "#000000",    # black
    "CD8": "#c9daf8",        # Weak blue
    "negative": "#e6e6e6"    # Light grey
}

# Step 8: Create stacked bar plot
ax = merged_matrix.plot(
    kind="bar",
    stacked=True,
    color=[colors[cat] for cat in merged_matrix.columns],
    figsize=(8, 6)
)

# Final formatting
ax.set_title("cd4cd8 Assignment by Compartment and Method")
ax.set_ylabel("Cell Count")
ax.tick_params(axis='x', rotation=45)
ax.legend(title="Assignment", bbox_to_anchor=(1.025, 0.75), loc='upper left')
plt.tight_layout()
plt.show()

# Step 4: Error correcting individual clones (i.e. Majority voting)
Aim is to correct  all negative cells, as well as wrongly labelled CD4 and CD8 T cells, while leaving CD4/CD8 untouched. 

In [ ]:
### Work with the right data - only T cells are allowed - no myeloid or myeoloid doublets! 
sdata_T = sdata.copy()
mu.pp.filter_obs(sdata_T["gex"], "L1_ann", lambda x: (x.isin(["T cells", "MAIT cells"]))) 
mu.pp.filter_obs(sdata_T["gex"], "clone_id", lambda x: x != "NaN")

CD4_TCRs = set(
    sdata_T["gex"].obs[
        (sdata_T["gex"].obs["cd4cd8_combined_assignment"].isin(["CD4"])) &
        (sdata_T["gex"].obs["clone_handle"].notna())
    ]["clone_handle"]
)

CD8_TCRs = set(
    sdata_T["gex"].obs[
        (sdata_T["gex"].obs["cd4cd8_combined_assignment"].isin(["CD8"])) &
        (sdata_T["gex"].obs["clone_handle"].notna())
    ]["clone_handle"]
)

# Theoretically, the most important cells to correct are the cells that occur in both CD4 and CD8 compartments
clonal_overlap_CD4CD8 =  CD4_TCRs & CD8_TCRs

print("In total:", len(clonal_overlap_CD4CD8), " Clones occur in both CD4 and CD8 assignment clone lists.\nThese are the clones I need to perform majority voting for")
print("  ")
print("This is how those cells are distributed:")

confusion_matrix = pd.crosstab(
    sdata_T["gex"].obs[sdata_T["gex"].obs["clone_handle"].isin(clonal_overlap_CD4CD8)].drop_duplicates()['cd4cd8_combined_assignment'],
    sdata_T["gex"].obs[sdata_T["gex"].obs["clone_handle"].isin(clonal_overlap_CD4CD8)].drop_duplicates()['cd4cd8_gex_assignment'],
    rownames=['cd4cd8_combined_assignment'],
    colnames=['cd4cd8_gex_assignment']
)

In [ ]:
print("These are example clones")
print(sdata_T["gex"].obs[sdata_T["gex"].obs["clone_handle"].isin(clonal_overlap_CD4CD8)][["clone_handle", "cd4cd8_combined_assignment"]].value_counts().head(5))
confusion_matrix

In [ ]:
sdata_T["gex"].obs[sdata_T["gex"].obs["clone_handle"].isin(["Pt12-16884"])][["CD8A_expr", "CD8B_expr", "CD4_expr", "cd4cd8_cs_assignment", "cd4cd8_combined_assignment"]].sort_values("CD4_expr", ascending=False).head(5)

In [ ]:
# Define clone ids seen in both CD4 and CD8 compartments  
sdata_T["gex"].obs["cd4cd8_cOverlap"] = pd.Categorical(
    np.where(sdata_T["gex"].obs["clone_handle"].isin(clonal_overlap_CD4CD8), "Yes", "No"))

df_input = sdata_T["gex"].obs[(sdata_T["gex"].obs["cd4cd8_cOverlap"] == "Yes")].sort_values("clone_handle")[["clone_handle", "cd4cd8_gex_assignment", "cd4cd8_combined_assignment"]]

conf_combined = pd.crosstab(
    df_input['cd4cd8_combined_assignment'],
    df_input['clone_handle'],
    rownames=['cd4cd8_combined_assignment'],
    colnames=['clone_handle']
)

print("A voting table per clonotype")
conf_combined

In [ ]:
### THEORETICALLY, but, extending the vote to all clones would enable us to correct more negative cells 

# I therefore redefine the inclusion criteria to "OR"
# Define clone ids seen in both CD4 and CD8 compartments  
sdata_T["gex"].obs["cd4cd8_cOverlap"] = pd.Categorical(
    np.where(sdata_T["gex"].obs["clone_handle"].isin(CD4_TCRs | CD8_TCRs), "Yes", "No"))

df_input = sdata_T["gex"].obs[(sdata_T["gex"].obs["cd4cd8_cOverlap"] == "Yes")].sort_values("clone_handle")[["clone_handle", "cd4cd8_gex_assignment", "cd4cd8_combined_assignment"]]

conf_combined = pd.crosstab(
    df_input['cd4cd8_combined_assignment'],
    df_input['clone_handle'],
    rownames=['cd4cd8_combined_assignment'],
    colnames=['clone_handle']
)


## Transform to a long format
## I like using stack here to generate a "multi-column index" similar to how you read R data frames. https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.stack.html#pandas.DataFrame.stack
df_long = conf_combined.stack().reset_index()
df_long.columns = ['cd4cd8_combined_assignment', 'clone_handle', 'cell_count']

## First - remove the negative and CD4/CD8- they do not provide information useful for correction
df_long = df_long[df_long["cd4cd8_combined_assignment"] != "negative"]
df_long = df_long[df_long["cd4cd8_combined_assignment"] != "CD4/CD8"]

# Group by clone_handle and find max cell count per group
max_counts = df_long.groupby('clone_handle', observed=True)['cell_count'].transform('max')  #Transform is typically used to summarize data without changing the shape of the data frame  
    #returns a Series of the same length as the original DataFrame—but repeats the group's max value for each row in that group.

# Flag rows where cell_count equals the max for their clone_handle
df_long['is_max'] = df_long['cell_count'] == max_counts

# Count how many labels share the max count per clone_handle
tie_counts = df_long[df_long['is_max']].groupby('clone_handle', observed=True).size()

# Identify clone_handles with ties (more than one max count)
tied_clones = tie_counts[tie_counts > 1].index

# Now assign the majority_assignment
def resolve_majority(row):
    if row['clone_handle'] in tied_clones:
        return 'tied'
    return row['cd4cd8_combined_assignment']

# Filter to max-count rows only
df_majority = df_long[df_long['is_max']].copy()
df_majority['majority_assignment'] = df_majority.apply(resolve_majority, axis=1)

# Keep only one row per clone_handle for merging (drop duplicates)
df_majority = df_majority[['clone_handle', 'majority_assignment']].drop_duplicates()

# Merge back with original df_long
df_assigned_PBCSF = df_long.merge(df_majority, on='clone_handle', how='left')
df_assigned_PBCSF.sort_values(by='clone_handle', inplace=True)

# Unique clones
df_assigned_PBCSF.head()

In [ ]:
## Focus on correctable clones
df_assigned_PBCSF = df_assigned_PBCSF[df_assigned_PBCSF["is_max"] == True]
df_assigned_PBCSF = df_assigned_PBCSF[["clone_handle", "cell_count", "majority_assignment"]].drop_duplicates().sort_values(by="cell_count", ascending=False)

print("Number of clones")
df_assigned_PBCSF["majority_assignment"].value_counts()

In [ ]:
#clean the dataframe for merging back unto the sc-seq.obs
df_assigned_PBCSF = df_assigned_PBCSF[["clone_handle", "majority_assignment"]].drop_duplicates()
CD4_ids = df_assigned_PBCSF[df_assigned_PBCSF["majority_assignment"] == "CD4"]["clone_handle"].tolist()
CD8_ids = df_assigned_PBCSF[df_assigned_PBCSF["majority_assignment"] == "CD8"]["clone_handle"].tolist()
tied_ids = df_assigned_PBCSF[df_assigned_PBCSF["majority_assignment"] == "tied"]["clone_handle"].tolist()

#Assign the results into a dataframe
## We only need pure CD4 and CD8 single positive labels. CD4/CD8 double positive labels will not be used. 
sdata["gex"].obs["cd4cd8_vote"] = np.where(sdata["gex"].obs["clone_handle"].isin(CD4_ids), "CD4", 
                                           np.where(sdata["gex"].obs["clone_handle"].isin(CD8_ids), "CD8", 
                                                    np.where(sdata["gex"].obs["clone_handle"].isin(tied_ids), "tied", "clone_not_neglibile_for_vote")))

sdata["gex"].obs[["cd4cd8_vote", "clone_handle"]].drop_duplicates()["cd4cd8_vote"].value_counts()

In [ ]:
pd.crosstab(
    sdata["gex"].obs["cd4cd8_vote"],
    sdata["gex"].obs["L1_ann"]
)

In [ ]:
pd.crosstab(
    sdata["gex"].obs["cd4cd8_vote"],
    sdata["gex"].obs["cd4cd8_combined_assignment"]
)

In [ ]:
#Update the cd4cd8_vote with information from previous assignment _combined_


## Option 1: Correct DPs using majority voting
sdata["gex"].obs["cd4cd8_vote"] = np.where(sdata["gex"].obs["L1_ann"] != "T cells", sdata["gex"].obs["L1_ann"],                                                                   ## If not a T cell - continue
                                  np.where(sdata["gex"].obs["cd4cd8_combined_assignment"] == sdata["gex"].obs["cd4cd8_vote"], sdata["gex"].obs["cd4cd8_combined_assignment"],     ## 1st priority - if labels are equal, change nothing. 
                                  np.where(sdata["gex"].obs["cd4cd8_combined_assignment"] != sdata["gex"].obs["cd4cd8_vote"],                                                     ## 3rd priority - if labels are different do something...   
                                     np.where(sdata["gex"].obs["cd4cd8_vote"] == "clone_not_neglibile_for_vote", sdata["gex"].obs["cd4cd8_combined_assignment"],                  ## ---vote = clone_not_neglibile_for_vote - use _comb_
                                     np.where(sdata["gex"].obs["cd4cd8_vote"] == "tied", sdata["gex"].obs["cd4cd8_combined_assignment"], sdata["gex"].obs["cd4cd8_vote"])),      ## ---vote = tied - use _comb_    ## Otherwise, use cd4cd8_vote as the final label.
                                           "error"))) 

## Option 2: Do not correct DPs
# sdata["gex"].obs["cd4cd8_vote"] = np.where(sdata["gex"].obs["L1_ann"] != "T cells", sdata["gex"].obs["L1_ann"],                                                                   ## If not a T cell - continue
#                                   np.where(sdata["gex"].obs["cd4cd8_combined_assignment"] == sdata["gex"].obs["cd4cd8_vote"], sdata["gex"].obs["cd4cd8_combined_assignment"],     ## 1st priority - if labels are equal, change nothing. 
#                                   np.where(sdata["gex"].obs["cd4cd8_combined_assignment"] == "CD4/CD8", sdata["gex"].obs["cd4cd8_combined_assignment"],                           ## 2nd priority - if _combined_ is double positive, change nothing. 
#                                   np.where(sdata["gex"].obs["cd4cd8_combined_assignment"] != sdata["gex"].obs["cd4cd8_vote"],                                                     ## 3rd priority - if labels are different do something...   
#                                      np.where(sdata["gex"].obs["cd4cd8_vote"] == "clone_not_neglibile_for_vote", sdata["gex"].obs["cd4cd8_combined_assignment"],                  ## ---vote = clone_not_neglibile_for_vote - use _comb_
#                                      np.where(sdata["gex"].obs["cd4cd8_vote"] == "tied", sdata["gex"].obs["cd4cd8_combined_assignment"], sdata["gex"].obs["cd4cd8_vote"])),      ## ---vote = tied - use _comb_    ## Otherwise, use cd4cd8_vote as the final label.
#                                            "error"))))                                            
                                                                        
                                                                                                                             

sdata["gex"].obs[["cd4cd8_combined_assignment","cd4cd8_vote"]].value_counts()

pd.crosstab(
    sdata["gex"].obs['cd4cd8_combined_assignment'],
    sdata["gex"].obs['cd4cd8_vote'],
    rownames=['cd4cd8_combined_assignment'],
    colnames=['cd4cd8_vote']
)

In [ ]:
plt.rcParams["figure.figsize"] = (3.5, 3)
sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_vote"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_vote"],
    groups=["CD4"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_vote"],
    groups=["CD8"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_vote"],
    groups=["CD4/CD8"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)

sc.pl.umap(
    sdata["gex"],
    color=["cd4cd8_vote"],
    groups=["negative"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=1
)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Summarize your progress in voting for cell identities

In [ ]:
### Work with the right data - only T cells are allowed - no myeloid or myeoloid doublets! 
sdata_T = sdata.copy()
mu.pp.filter_obs(sdata_T["gex"], "L1_ann", lambda x: (x.isin(["T cells"]))) 
mu.pp.filter_obs(sdata_T["gex"], "clone_id", lambda x: x != "NaN")

# Step 1: Build both confusion matrices
confusion_matrix_gex = pd.crosstab(
    sdata_T["gex"].obs['cd4cd8_gex_assignment'],
    sdata_T["gex"].obs['Compartment'],
    rownames=['cd4cd8_gex_assignment'],
    colnames=['Compartment']
)

confusion_matrix_combined = pd.crosstab(
    sdata_T["gex"].obs['cd4cd8_combined_assignment'],
    sdata_T["gex"].obs['Compartment'],
    rownames=['cd4cd8_combined_assignment'],
    colnames=['Compartment']
)

confusion_matrix_vote = pd.crosstab(
    sdata_T["gex"].obs['cd4cd8_vote'],
    sdata_T["gex"].obs['Compartment'],
    rownames=['cd4cd8_vote'],
    colnames=['Compartment']
)

# Step 2: Ensure consistent ordering of assignment categories
assignments = ["CD4", "CD4/CD8", "CD8", "negative"]
confusion_matrix_gex = confusion_matrix_gex.reindex(assignments).fillna(0)
confusion_matrix_combined = confusion_matrix_combined.reindex(assignments).fillna(0)
confusion_matrix_vote = confusion_matrix_vote.reindex(assignments).fillna(0)

# Step 3: Add suffixes to columns to differentiate gex/combined
confusion_matrix_gex.columns = [f"{col}_gex" for col in confusion_matrix_gex.columns]
confusion_matrix_combined.columns = [f"{col}_combined" for col in confusion_matrix_combined.columns]
confusion_matrix_vote.columns = [f"{col}_vote" for col in confusion_matrix_vote.columns]

# Step 4: Concatenate both matrices column-wise
merged_matrix = pd.concat([confusion_matrix_gex, confusion_matrix_combined, confusion_matrix_vote], axis=1)

# Step 5: Transpose for plotting (samples on x-axis)
merged_matrix = merged_matrix.T  # rows = compartment_type, columns = assignment types

# 👉 Step 6: Sort index so CSF_x come first
merged_matrix = merged_matrix.sort_index(key=lambda x: x.str.contains("CSF"), ascending=False)

# Step 7: Plot with consistent grey-scale colors
colors = {
    "CD4": "#f4cccc",        # Weak red
    "CD4/CD8": "#000000",    # black
    "CD8": "#c9daf8",        # Weak blue
    "negative": "#e6e6e6"    # Light grey
}

# Step 8: Create stacked bar plot
ax = merged_matrix.plot(
    kind="bar",
    stacked=True,
    color=[colors[cat] for cat in merged_matrix.columns],
    figsize=(8, 6)
)

# Final formatting
ax.set_title("cd4cd8 Assignment by Compartment and Method - T cells only")
ax.set_ylabel("Cell Count")
ax.tick_params(axis='x', rotation=45)
ax.legend(title="Assignment", bbox_to_anchor=(1.025, 0.75), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
sdata.update()

## Consistency check
How much of each co-receptor du the different categories of cd4cd8_vote express?

In [ ]:
coreceptors = {
    "T-cell status": ["CD3E", "CD4", "CD8A", "CD8B"]
}

sc.pl.dotplot(sdata["gex"], coreceptors, groupby="cd4cd8_vote", standard_scale="var", color_map='RdYlBu_r')


# Step 5: Assign each cluster based on CD4 and CD8 frequencies

### The main issue now is, that CSF contains a lot of negative cells. 
#### Summary stats

In [ ]:
sdata["gex"].obs[["cd4cd8_vote", "Compartment"]]

# Create confusion matrix
confusion_matrix = pd.crosstab(sdata["gex"].obs['cd4cd8_vote'], sdata["gex"].obs['Compartment'], rownames=['cd4cd8_vote'], colnames=['Compartment'])

print(confusion_matrix)

In [ ]:
# # Filter the dataframe where Compartment == "CSF" and L1_ann == "T cells"
# filtered_df = sdata["gex"].obs[
#     (sdata["gex"].obs["Compartment"] == "CSF")
# ]
# filtered_df["Compartment"].value_counts()

# # Build confusion matrix
# confusion_matrix = pd.crosstab(
#     filtered_df['L1_ann'],
#     filtered_df['cd4cd8_vote'],
#     rownames=['leiden_4.5'],
#     colnames=['Combined assessment']
# )

# # Calculate **row-wise percentages**
# percentage_matrix = confusion_matrix.div(confusion_matrix.sum(axis=1), axis=0) * 100

# # Normalize **per row** for color scale
# normalized_matrix = percentage_matrix.copy()
# for row in normalized_matrix.index:
#     row_min = normalized_matrix.loc[row, :].min()
#     row_max = normalized_matrix.loc[row, :].max()
#     if row_max != row_min:  # Avoid division by zero
#         normalized_matrix.loc[row, :] = (normalized_matrix.loc[row, :] - row_min) / (row_max - row_min)
#     else:
#         normalized_matrix.loc[row, :] = 0.5  # If constant row, set mid-range

# # Plot using normalized matrix for colors, but **row-wise percentages** for annotations
# plt.figure(figsize=(5, 5))
# ax = sb.heatmap(
#     normalized_matrix,
#     annot=percentage_matrix.round(1),
#     fmt=".1f",
#     cmap="Blues",
#     cbar=True
# )

# # Keep x-axis ticks at bottom
# ax.tick_params(axis='x', labelbottom=True, labeltop=False)

# # Labels and title
# ax.set_xlabel("Combined, voted Assessment")
# ax.set_ylabel("")
# plt.title("For CSF T cells\nPercentage of cells assigned (row-wise normalization)", pad=40)

# plt.show()

### Subcluster CSF T cells

In [ ]:
csf_T = sdata.copy()
mu.pp.filter_obs(csf_T["gex"], "Compartment", lambda x: (x.isin(["CSF"]))) 
mu.pp.filter_obs(csf_T["gex"], "L1_ann", lambda x: (x.isin(["T cells"]))) 
csf_T.update()


In [ ]:
## Filter genes away that are present in very few cells
## Threshold from: https://muon-tutorials.readthedocs.io/en/latest/cite-seq/1-CITE-seq-PBMC-5k.html
mu.pp.filter_var(csf_T["gex"], 'n_cells_by_counts', lambda x: x >= 3)
csf_T.update()

In [ ]:
## Renormalize after subsetting on specific genes. 
csf_T["gex"].X = csf_T["gex"].layers["counts"].copy()
csf_T["gex"].uns.pop("log1p", None) #Removes log1P if it exists
sc.pp.normalize_total(csf_T["gex"], target_sum=1e4, exclude_highly_expressed=False)   #1e4 is important. Do not change
sc.pp.log1p(csf_T["gex"])

In [ ]:
calculate_leiden(csf_T["gex"], resolution=2, n_high_var=5000, remove_tcr_genes=True)   
calculate_umap(csf_T["gex"], n_high_var=5000, remove_tcr_genes=True)

In [ ]:
sc.pl.umap(csf_T["gex"], color=["leiden_2", "cd4cd8_vote", "chain_pairing", "FOXP3"], legend_loc="on data", cmap="RdBu_r", size=20,legend_fontweight= 'bold', frameon= False , vmin=0, legend_fontoutline=2)

In [ ]:
# Build confusion matrix
confusion_matrix = pd.crosstab(
    csf_T["gex"].obs['leiden_2'],
    csf_T["gex"].obs['chain_pairing'],
)

# Calculate row-wise percentages
percentage_matrix = confusion_matrix.div(confusion_matrix.sum(axis=1), axis=0) * 100

# Plot heatmap with global color scale from 0 to 100
plt.figure(figsize=(6, 15))
ax = sb.heatmap(
    percentage_matrix,
    annot=percentage_matrix.round(1),
    fmt=".1f",
    cmap="Blues",
    vmin=0,
    vmax=100,
    cbar=True
)

# Keep x-axis ticks at both top and bottom
ax.tick_params(axis='x', labelbottom=True, labeltop=True)

# Axis labels and title
ax.set_xlabel("Chain_pairing")
ax.set_ylabel("Leiden 2")
plt.title("Row-wise percentages (color scale 0-100)", pad=40)

plt.show()

In [ ]:
# Build confusion matrix
confusion_matrix = pd.crosstab(
    csf_T["gex"].obs['leiden_2'],
    csf_T["gex"].obs['cd4cd8_vote'],
)

# Calculate row-wise percentages
percentage_matrix = confusion_matrix.div(confusion_matrix.sum(axis=1), axis=0) * 100

# Plot heatmap with global color scale from 0 to 100
plt.figure(figsize=(5, 15))
ax = sb.heatmap(
    percentage_matrix,
    annot=percentage_matrix.round(1),
    fmt=".1f",
    cmap="Blues",
    vmin=0,
    vmax=100,
    cbar=True
)

# Keep x-axis ticks at both top and bottom
ax.tick_params(axis='x', labelbottom=True, labeltop=True)

# Axis labels and title
ax.set_xlabel("Combined, voted assessment")
ax.set_ylabel("Leiden 2")
plt.title("Row-wise percentages (color scale 0-100)", pad=40)

plt.show()

### CSF T cells - CD4 and CD8 assignment
Cluster 17 is the only cluster that shows a considerable concentration of CD4/CD8

In [ ]:
## Simple clusters to identify T cell versus myeloid cell
marker_genes = {
    "T cells": ["CD45RA_adt","CD62L_adt", "CD95_adt","CD8_adt","CD4_adt", "CD3E", "CD8A", "CD8B", "CD4"],
    "Effector": ["CCL5", "GZMK", "GZMB", "GNLY", "NKG7", "LAG3", "TIGIT", "PDCD1"],
    "IFN-sig": ["IFI44", "IFI44L", "ISG15", "OAS2"], 
    "T naive": ["LEF1", "CCR7", "TCF7", "SELL"],
    "T reg": ["FOXP3", "IL2RA", "CTLA4"],
    "Antigen-presentation": ["HLA-DRA", "HLA-DRB1", "CD74"],
    "NfKB" : ["NFKB2", "NFKBIA", "RELB","RELA"],
    "Ribosomal_genes": ["RPLP1", "EEF1A1", "RPL28"]
    # "AXL+SIGLEC6+ DCs":["PPP1R14A", "AXL", "SIGLEC6"],
}

sc.pl.dotplot(csf_T["gex"], marker_genes, groupby="leiden_2", standard_scale="var")



In [ ]:

plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(csf_T["gex"], color=["cd4cd8_vote"], groups="CD4", legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, size=15, legend_loc = "on data")
sc.pl.umap(csf_T["gex"], color=["cd4cd8_vote"], groups="CD8", legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, size=15, legend_loc = "on data")
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
annotation_dict = {
    #0, 7, 12, 13, 15, 18 are CD8 T cells
    #14 and 17 are mixed
    #14 are 39.9% CD8, and 24.2% CD4. 
    ## Expression level for CD4 appear low here, yet CD8 remains very high. Annotating negatives as CD8
    '0': 'CD8 T cells',
    '1': 'CD4 T cells',
    '2': 'CD4 T cells',
    '3': 'CD4 T cells',
    '4': 'CD4 T cells',  
    '5': 'CD4 T cells',
    '6': 'CD4 T cells',
    '7': 'CD8 T cells',
    '8': 'CD4 T cells',
    '9': 'CD4 T cells',
    '10': 'CD4 T cells',
    '11': 'CD4 T cells',
    '12': 'CD8 T cells',
    '13': 'CD8 T cells',
    '14': 'CD8 T cells',
    '15': 'CD8 T cells',
    '16': 'CD4 T cells',
    '17': 'CD4/CD8 T cells',
    '18': 'CD8 T cells',
    '19': 'CD4 T cells',
    '20': 'CD4 T cells',
    '21': 'CD4 T cells',
}
 
 
# Add cell type column based on annotation
csf_T["gex"].obs['L2_ann'] = [annotation_dict[sample] for sample in csf_T["gex"].obs['leiden_2']]

## Update L2_ann so that it retains cd4cd8_vote information

In [ ]:
# Practically, only correct the negatives we could not discover the identity of before. 
csf_T["gex"].obs["L2.5_ann"] = pd.Categorical(np.where(csf_T["gex"].obs["cd4cd8_vote"] == "negative", csf_T["gex"].obs["L2_ann"], csf_T["gex"].obs["cd4cd8_vote"])) 

#Fix the labels from cd4cd8_vote
csf_T["gex"].obs["L2.5_ann"] = pd.Categorical(np.where(csf_T["gex"].obs["L2.5_ann"] == "CD4", "CD4 T cells", 
                                                       np.where(csf_T["gex"].obs["L2.5_ann"] == "CD8", "CD8 T cells", 
                                                                 np.where(csf_T["gex"].obs["L2.5_ann"] == "CD4/CD8", "CD4/CD8 T cells", csf_T["gex"].obs["L2.5_ann"]))))

In [ ]:
## Visualize
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(csf_T["gex"], color=["leiden_2", "CD8A", "CD4", "cd4cd8_vote", "L2_ann", "L2.5_ann"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, legend_fontoutline=2, size=15, legend_loc = "on data")
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

## Transfer annotations made in CSF to global sdata

In [ ]:
sdata["gex"].obs["cd4cd8_vote"].value_counts()

In [ ]:
#Create an index dictionary with annotations you want to transfer to another AnnData object
obs_df = csf_T["gex"].obs.copy()
index_dictionary = obs_df["L2.5_ann"].to_dict()

#Create the L2.5_ann column in the new AnnData object by mapping index and value pairs from the index_indictionary
sdata["gex"].obs["L2.5_ann"] = sdata["gex"].obs.index.map(index_dictionary)

#In case of NAN repalce L2.5_ann with L1_ann or cd4cd8_vote
sdata["gex"].obs["L2.5_ann"] = pd.Categorical(np.where(sdata["gex"].obs["Compartment"] == "PB", sdata["gex"].obs["cd4cd8_vote"],                                               #If PB - keep labels
                                              np.where((sdata["gex"].obs["Compartment"] == "CSF") & (sdata["gex"].obs["L1_ann"]=="T cells"), sdata["gex"].obs["L2.5_ann"],     #If CSF + T cells - use 
                                              np.where((sdata["gex"].obs["Compartment"] == "CSF") & (sdata["gex"].obs["L1_ann"]!="T cells"), sdata["gex"].obs["cd4cd8_vote"],       #If CSF NOT T cells - use old label
                                                       sdata["gex"].obs["cd4cd8_vote"]))))

#Fix the labels from cd4cd8_vote
sdata["gex"].obs["L2.5_ann"] = pd.Categorical(np.where(sdata["gex"].obs["L2.5_ann"] == "CD4", "CD4 T cells", 
                                                       np.where(sdata["gex"].obs["L2.5_ann"] == "CD8", "CD8 T cells", 
                                                                 np.where(sdata["gex"].obs["L2.5_ann"] == "CD4/CD8", "CD4/CD8 T cells", sdata["gex"].obs["L2.5_ann"]))))

In [ ]:
pd.crosstab(
    sdata["gex"].obs["L2.5_ann"],
    sdata["gex"].obs["cd4cd8_vote"]
)

In [ ]:
## Visualize
plt.rcParams["figure.figsize"] = (5, 5)
sc.pl.umap(sdata["gex"], color=["leiden_4.5", "cd4cd8_vote", "L2.5_ann"], legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, size=5, legend_loc = "on data")
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Double negative clones according to gene expression in CSF
Double negative cells in autoimmune diseases such as SLE are thought to derive from mature CD8 T cells. 
- Review: https://pmc.ncbi.nlm.nih.gov/articles/PMC8018563/
- Primary evidence: https://www.nature.com/articles/s41467-020-16636-4

In [ ]:
### ----------
## It's really interesting that certain clones are fully negative.
# obs_df = sdata["gex"].obs.copy()

# # Count the total number of cells for each clone in CSF
# obs_df = obs_df[obs_df["Compartment"] == "CSF"]
# obs_df["total"] = obs_df.groupby("clone_handle").transform("size")

# # Count the number of cells that were negative for co-receptor expression for each cell
# obs_df = obs_df[(obs_df["cd4cd8_gex_assignment"]=="negative") & (obs_df["L1_ann"]== "T cells")]
# obs_df = obs_df[(~obs_df["donor"].isin(["Pt4", "Pt5"])) & (obs_df["L1_ann"]== "T cells")]
# obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories()
# obs_df = obs_df.groupby(["donor", "clone_handle", "catpat", "clone_size", "total"], observed=True).size().reset_index(name="count").sort_values("count", ascending=False)
# ### ----------

### ----------
### But the easier-to-communicate metric is the percentage of cells that are negative for a co-receptor
## It's really interesting that certain clones are fully negative.
obs_df = sdata["gex"].obs.copy()

# Count the total number of cells for each clone in CSF
obs_df = obs_df[(obs_df["Compartment"] == "CSF") & (obs_df["L1_ann"] == "T cells") & (~obs_df["donor"].isin(["Pt4", "Pt5"]))]
obs_df = obs_df[~obs_df["clone_handle"].isna()]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories()
obs_df["total_per_clone"] = obs_df.groupby(["clone_handle", "donor"], observed=True).transform("size")

# Select columns
obs_df = obs_df[["donor", "catpat", "clone_handle", "total_per_clone"]]

# Select distributions --- Create distributions for total cells observed
# Work with expanded cells in CSF
df_all = obs_df[obs_df["total_per_clone"] > 0].copy()
df5 = obs_df[obs_df["total_per_clone"] > 5].copy()
df10 = obs_df[obs_df["total_per_clone"] > 10].copy()
df15 = obs_df[obs_df["total_per_clone"] > 15].copy()
df20 = obs_df[obs_df["total_per_clone"] > 20].copy()

df_all["donor_total_cells"] = df_all.groupby("donor")["total_per_clone"].transform("sum")
df5["donor_total_cells"] = df5.groupby("donor")["total_per_clone"].transform("sum")
df10["donor_total_cells"] = df10.groupby("donor")["total_per_clone"].transform("sum")
df15["donor_total_cells"] = df15.groupby("donor")["total_per_clone"].transform("sum")
df20["donor_total_cells"] = df20.groupby("donor")["total_per_clone"].transform("sum")

# Work with expanded cells
df_all["source"] = "All cells"
df5["source"] = "5"
df10["source"] = "10"
df15["source"] = "15"
df20["source"] = "20"

concat_1 = pd.concat([df_all, df5, df10, df15, df20])
concat_1 = concat_1[["donor", "catpat", "donor_total_cells", "source"]].drop_duplicates()

In [ ]:
# Create distributions for negative cells observed in CSF ----
obs_df = sdata["gex"].obs.copy()

# Count the total number of cells for each clone in CSF
obs_df = obs_df[(obs_df["Compartment"] == "CSF") & (obs_df["L1_ann"] == "T cells") & (obs_df["cd4cd8_gex_assignment"] == "negative") & (~obs_df["donor"].isin(["Pt4", "Pt5"]))]
obs_df = obs_df[~obs_df["clone_handle"].isna()]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories()
obs_df["total_per_clone"] = obs_df.groupby(["clone_handle", "donor"], observed=True).transform("size")
obs_df = obs_df[["donor", "catpat", "clone_handle", "total_per_clone"]]

# Thresholds
neg_df_all = obs_df[obs_df["total_per_clone"] > 0].copy()
neg_df5 = obs_df[obs_df["total_per_clone"] > 5].copy()
neg_df10 = obs_df[obs_df["total_per_clone"] > 10].copy()
neg_df15 = obs_df[obs_df["total_per_clone"] > 15].copy()
neg_df20 = obs_df[obs_df["total_per_clone"] > 20].copy()

neg_df_all["negative_cells"] = neg_df_all.groupby("donor")["total_per_clone"].transform("sum")
neg_df5["negative_cells"] = neg_df5.groupby("donor")["total_per_clone"].transform("sum")
neg_df10["negative_cells"] = neg_df10.groupby("donor")["total_per_clone"].transform("sum")
neg_df15["negative_cells"] = neg_df15.groupby("donor")["total_per_clone"].transform("sum")
neg_df20["negative_cells"] = neg_df20.groupby("donor")["total_per_clone"].transform("sum")

# Work with expanded cells
neg_df_all["source"] = "All cells"
neg_df5["source"] = "5"
neg_df10["source"] = "10"
neg_df15["source"] = "15"
neg_df20["source"] = "20"

concat_2 = pd.concat([neg_df_all, neg_df5, neg_df10, neg_df15, neg_df20])
concat_2 = concat_2[["donor", "catpat", "negative_cells", "source"]].drop_duplicates()

#left join concat_2 onto conat_1
plot_df = concat_1.merge(concat_2, on=["donor", "catpat", "source"], how="left")
plot_df["negative_cells"]=plot_df["negative_cells"].fillna(0)

## Percentage
plot_df["percentage"] = 100* plot_df["negative_cells"] / plot_df["donor_total_cells"]

set_df = neg_df10[["donor", "catpat", "clone_handle"]].value_counts().reset_index()
set_df = set_df[set_df["count"] > 0]["clone_handle"].tolist()
set_df


In [ ]:
# Complete combinations of donor and source
complete_df = (
    pd.MultiIndex.from_product(
        [plot_df["donor"].unique(), plot_df["source"].unique()],
        names=["donor", "source"]
    )
    .to_frame(index=False)
    .merge(plot_df, on=["donor", "source"], how="left")
)

# Complete catpat
donor_to_catpat = (
    plot_df[["donor", "catpat"]]
    .drop_duplicates()
    .set_index("donor")["catpat"]
    .to_dict()
)

# # replace NaN for complete_df["catpat"] with values from the donor_to_catpat dictionary
complete_df["catpat"] = complete_df["catpat"].fillna(
    complete_df["donor"].map(donor_to_catpat)
)

# # Fill NaNs for percentage
complete_df["percentage"] = complete_df["percentage"].fillna(0)
# complete_df = complete_df[complete_df["source"] != "All cells"]

plt.figure(figsize=(4, 4))

sb.boxplot(
    data=complete_df,
    x="source",
    y="percentage",
    hue="catpat",
    order=["All cells", "5", "10", "15", "20"],
    dodge=True,
    showfliers=False,
    fill=False
)

sb.stripplot(
    data=complete_df,
    x="source",
    y="percentage",
    hue="catpat",
    order=["All cells", "5", "10", "15", "20"],
    dodge=True,
    jitter=0.2,
    alpha=0.75,
    legend=False
)

# Keep only one set of legend entries
handles, labels = ax.get_legend_handles_labels()
n_hue = plot_df["catpat"].nunique()

ax.legend(
    handles[:n_hue],
    labels[:n_hue],
    title="catpat",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)

plt.ylabel("Percentage")
plt.xlabel("Expansion threshold")
sb.despine()
plt.grid(False)
plt.show()

## What cluster do DN cells belong to in CSF?

In [ ]:
csf_T["gex"].obs = csf_T["gex"].obs.copy()

csf_T["gex"].obs["Expanded_DN_boolean"]= np.where(csf_T["gex"].obs["clone_handle"].isin(set_df), "Expanded_DN", "Background")
csf_T["gex"].obs["Expanded_DN_clone"]= np.where(csf_T["gex"].obs["clone_handle"].isin(set_df), csf_T["gex"].obs["clone_handle"], pd.NA)

In [ ]:
## Visualize
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(csf_T["gex"], color=["leiden_2", "L2.5_ann", "PRF1"], legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, size=15, cmap="RdBu_r", legend_loc = "on data")
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
## Visualize
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(csf_T["gex"], color=["Expanded_DN_boolean"], groups="Expanded_DN",legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, size=15)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
## Visualize
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(csf_T["gex"], color=["Expanded_DN_clone"], groups=["Pt12-16973", "Pt12-16884", "Pt11-5914", "Pt11-7652"], legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, size=50)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
# Make the figure wider using figsize
with plt.rc_context({"figure.figsize": (10, 2)}):
    sc.pl.violin(
        csf_T["gex"], 
        keys=["GZMB"],
        groupby="Expanded_DN_clone",
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')  # fix alignment under ticks
plt.legend().set_visible(False)
plt.tight_layout()                   # adjust spacing if needed
plt.show()

In [ ]:
# Make the figure wider using figsize
with plt.rc_context({"figure.figsize": (10, 2)}):
    sc.pl.violin(
        csf_T["gex"], 
        keys=["GZMK"],
        groupby="Expanded_DN_clone",
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')  # fix alignment under ticks
plt.legend().set_visible(False)
plt.tight_layout()                   # adjust spacing if needed
plt.show()

In [ ]:
# Make the figure wider using figsize
with plt.rc_context({"figure.figsize": (10, 2)}):
    sc.pl.violin(
        csf_T["gex"], 
        keys=["PRF1"],
        groupby="Expanded_DN_clone",
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')  # fix alignment under ticks
plt.legend().set_visible(False)
plt.tight_layout()                   # adjust spacing if needed
plt.show()

In [ ]:
# Make the figure wider using figsize
with plt.rc_context({"figure.figsize": (10, 2)}):
    sc.pl.violin(
        csf_T["gex"], 
        keys=["FOXP3"],
        groupby="Expanded_DN_clone",
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')  # fix alignment under ticks
plt.legend().set_visible(False)
plt.tight_layout()                   # adjust spacing if needed
plt.show()

In [ ]:
## Visualize
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(csf_T["gex"], color=["Expanded_DN_clone"], groups=["Pt11-5914", "Pt11-6555", "Pt14-25573"], legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, size=50)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

### Annotate clonal overlap, CD4 v CD8
#### These are the wierd double negative clones especially! 

In [ ]:
sdata.update()

In [ ]:
cd8_dat = sdata["gex"].obs[sdata["gex"].obs["L2.5_ann"] == "CD8 T cells"]["clone_handle"].drop_duplicates().tolist()
cd4_dat = sdata["gex"].obs[sdata["gex"].obs["L2.5_ann"] == "CD4 T cells"]["clone_handle"].drop_duplicates().tolist()

CD8_TCRs = set(cd8_dat)
CD4_TCRs = set(cd4_dat)

set1 = CD4_TCRs
set2 = CD8_TCRs

# Compute sizes for each section
only_set1 = len(set1 - set2)
only_set2 = len(set2 - set1)
intersection = len(set1 & set2)

total = only_set1 + only_set2 + intersection

# Compute percentages
pct1 = 100 * only_set1 / total
pct2 = 100 * only_set2 / total
pct_inter = 100 * intersection / total

# Create a Venn diagrams
vennAll = venn2(subsets=(only_set1, only_set2, intersection), set_labels=('CD4 TCRs', 'CD8 TCRs'))

# Modify labels to show percentages
if vennAll.get_label_by_id('10'):
    vennAll.get_label_by_id('10').set_text(f"{only_set1}\n({pct1:.1f}%)")
if vennAll.get_label_by_id('01'):
    vennAll.get_label_by_id('01').set_text(f"{only_set2}\n({pct2:.1f}%)")
if vennAll.get_label_by_id('11'):
    vennAll.get_label_by_id('11').set_text(f"{intersection}\n({pct_inter:.1f}%)")

# Set colors to a colorblind-friendly palette
vennAll.get_patch_by_id('10').set_color('#ff0000')  # left = Red
vennAll.get_patch_by_id('01').set_color('#1f77b4')  # right = Blue
vennAll.get_patch_by_id('11').set_color('#ff7f0e')  # overlap = orange

# Adjust transparency for all patches
for venn in [vennAll]:
    for patch in venn.patches:
        if patch:
            patch.set_alpha(0.7)

# Display the plot
plt.title("Clonotype overlap, CD4 vs CD8") 
plt.tight_layout()
plt.show()

#2% overlap when CD4/CD8 T cells are included
#0.5% overlap when CD4 v CD8 are excluded

In [ ]:
## Who are the intersecting clones?
Intersecting_clones = set1 & set2
sdata["gex"].obs[sdata["gex"].obs["clone_handle"].isin(Intersecting_clones)][["clone_handle", "clone_size"]].sort_values(by = "clone_size", ascending=False).drop_duplicates().head(4)

In [ ]:
#Test problem clone
sdata["gex"].obs[sdata["gex"].obs["clone_handle"]=="Pt11-7652"][["clone_handle", "Compartment", "cd4cd8_gex_assignment" ,"cd4cd8_combined_assignment", "cd4cd8_vote", "L2.5_ann"]].value_counts().reset_index()

In [ ]:
# The problem clone is a DN clone expanded in a patient 11 
sdata["gex"].obs[sdata["gex"].obs["clone_handle"]=="Pt11-7652"][["clone_handle", "Compartment",  "L2.5_ann", "chain_pairing"]].value_counts().reset_index()

## Possibly autoreactive


## Majority voting for clones present in both CD4 and CD8
The logic is, that manual cluster annotation may contribute to noise. A cell belonging to a CD4 T-cell clone can fx occur in a CD8 T cell area. Manual cluster annotation would here lead to a false positive CD8 T cells. To mitigate that, we do a final round of voting. Besides, there are no clear MAIT cell clusters in CSF, yet the same MAIT-associated clones are present and should be annotated clearly in PB

In [ ]:
sdata["gex"].obs["L2.5_ann"].value_counts()

In [ ]:
### Work with the right data - only T cells are allowed - no myeloid or myeoloid doublets! 
obs_df = sdata["gex"].obs.copy()

obs_df = obs_df[obs_df["L2.5_ann"].isin(["CD4 T cells", "CD8 T cells", "CD4/CD8 T cells", "MAIT cells"])]
obs_df = obs_df[~obs_df["clone_id"].isna()]

# # sdata_T = sdata.copy()
# # mu.pp.filter_obs(sdata_T["gex"], "L2.5_ann", lambda x: (x.isin(["CD4 T cells", "CD8 T cells", "CD4/CD8 T cells", "MAIT cells"]))) 
# # mu.pp.filter_obs(sdata_T["gex"], "clone_id", lambda x: ~np.isnan(x))

# cd4_set = set(obs_df[obs_df["L2.5_ann"] == "CD4 T cells"]["clone_handle"])
# cd8_set = set(obs_df[obs_df["L2.5_ann"] == "CD8 T cells"]["clone_handle"])
# dp_set = set(obs_df[obs_df["L2.5_ann"] ==  "CD4/CD8 T cells"]["clone_handle"])
# mait_set = set(obs_df[obs_df["L2.5_ann"] == "MAIT cells"]["clone_handle"])

# # Compute sizes for each section
# intersection = cd4_set & cd8_set
# total_set = intersection.union(dp_set).union(mait_set)

# print("In total:", len(total_set), " clones among L2_ann_finn that occurs in either CD4/CD8 or are present in both CD4 and CD8 SP groups")
# print("  ")
# print("This is how those cells are distributed:")

# Define clone ids seen in both CD4 and CD8 compartments  
# obs_df["cOverlap2"] = pd.Categorical(
#     np.where(obs_df["clone_handle"].isin(total_set), "Yes", "No"))
# df_input = obs_df[(obs_df["cOverlap2"] == "Yes")][["clone_handle", "cd4cd8_vote", "L2.5_ann"]]

df_input = obs_df[["clone_handle", "L2.5_ann"]]

conf_combined = pd.crosstab(
    df_input['L2.5_ann'],
    df_input['clone_handle'],
    rownames=['L2.5_ann'],
    colnames=['clone_handle']
)

print("A voting table per clonotype")
conf_combined

In [ ]:
## Transform to a long format
## I like using stack here to generate a "multi-column index" similar to how you read R data frames. https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.stack.html#pandas.DataFrame.stack
df_long = conf_combined.stack().reset_index(name = "cell_count")
df_long.head(2)

In [ ]:
## First - remove the CD4/CD8 counts from majority voting considerations. Each clone appears to associaated with either CD4 or CD8 outside the CD4/CD8 clusters. 
### We assume that these cells exists as singlet CD4 or CD8 T cells naturally - but form interactions either as a result of biology or our methodology, and we don't know which one. 
df_long = df_long[df_long["L2.5_ann"] != "CD4/CD8 T cells"]   #Doublet identities are not a valid end identity, exclude it in the voting
df_long["L2.5_ann"] = df_long["L2.5_ann"].cat.remove_unused_categories()

# Group by clone_handle and find max cell count per group
max_counts = df_long.groupby('clone_handle', observed=True)['cell_count'].transform('max')  #Transform is typically used to summarize data without changing the shape of the data frame  
    #returns a Series of the same length as the original DataFrame—but repeats the group's max value for each row in that group.

# Flag rows where cell_count equals the max for their clone_handle
df_long['is_max'] = df_long['cell_count'] == max_counts

# Count how many labels share the max count per clone_handle
tie_counts = df_long[df_long['is_max']].groupby('clone_handle', observed=True).size()

# Identify clone_handles where there are more than 1 label that is "tied"
tied_clones = tie_counts[tie_counts > 1].index

# Now assign the majority_assignment
def resolve_majority(row):
    if row['clone_handle'] in tied_clones:
        return 'tied'
    return row['L2.5_ann']

# Filter to max-count rows only
df_majority = df_long[df_long['is_max']].copy()
df_majority['majority_assignment'] = df_majority.apply(resolve_majority, axis=1)

# Keep only one row per clone_handle for merging (drop duplicates)
df_majority = df_majority[['clone_handle', 'majority_assignment']].drop_duplicates()

# Merge back with original df_long
df_assigned_PBCSF = df_long.merge(df_majority, on='clone_handle', how='left')
df_assigned_PBCSF.sort_values(by='clone_handle', inplace=True)

In [ ]:
# ### SIDE INVESTIGATION - IF I CONSIDER CD4/CD8 T cells a separate subset. 

# # Check - cd4/cd8 majority votes
# df_assigned_PBCSF[(df_assigned_PBCSF["majority_assignment"] == "CD4/CD8 T cells") & (df_assigned_PBCSF["is_max"] == True)].sort_values("cell_count", ascending=False)

# df_obs = sdata["gex"].obs
# df_obs[df_obs["clone_handle"] == "Pt20-111698"].groupby(["L2.5_ann"], observed=True).size().reset_index(name="counts")

# df_obs = sdata["gex"].obs
# df_obs[df_obs["clone_handle"] == "Pt1-175"].groupby(["L2.5_ann"], observed=True).size().reset_index(name="counts")

# df_obs = sdata["gex"].obs
# df_obs[df_obs["clone_handle"] == "Pt1-33991"].groupby(["L2.5_ann"], observed=True).size().reset_index(name="counts")

# df_obs = sdata["gex"].obs
# df_obs[df_obs["clone_handle"] == "Pt20-112031"].groupby(["L2.5_ann"], observed=True).size().reset_index(name="counts")

In [ ]:
## Focus on correctable clones
df_assigned_PBCSF = df_assigned_PBCSF[df_assigned_PBCSF["is_max"] == True]
df_assigned_PBCSF = df_assigned_PBCSF[["clone_handle", "cell_count", "majority_assignment"]].drop_duplicates().sort_values(by="cell_count", ascending=False)

df_assigned_PBCSF["majority_assignment"].value_counts()

In [ ]:
#clean the dataframe for merging back unto the sc-seq.obs
df_assigned_PBCSF = df_assigned_PBCSF[["clone_handle", "majority_assignment"]].drop_duplicates()
CD4_ids = df_assigned_PBCSF[df_assigned_PBCSF["majority_assignment"] == "CD4 T cells"]["clone_handle"].tolist()
CD8_ids = df_assigned_PBCSF[df_assigned_PBCSF["majority_assignment"] == "CD8 T cells"]["clone_handle"].tolist()
tied_ids = df_assigned_PBCSF[df_assigned_PBCSF["majority_assignment"] == "tied"]["clone_handle"].tolist()
mait_ids = df_assigned_PBCSF[df_assigned_PBCSF["majority_assignment"] == "MAIT cells"]["clone_handle"].tolist()

#Assign the results into a dataframe
## We only need pure CD4 and CD8 single positive labels. CD4/CD8 double positive labels will not be used. 
sdata["gex"].obs["final_ann_vote"] = np.where(sdata["gex"].obs["clone_handle"].isin(CD4_ids), "CD4 T cells", 
                                           np.where(sdata["gex"].obs["clone_handle"].isin(CD8_ids), "CD8 T cells", 
                                                    np.where(sdata["gex"].obs["clone_handle"].isin(mait_ids), "MAIT cells", 
                                                    np.where(sdata["gex"].obs["clone_handle"].isin(tied_ids), "tied", "clone_not_neglibile_for_vote"))))

sdata["gex"].obs[["final_ann_vote", "clone_handle"]].drop_duplicates()["final_ann_vote"].value_counts()


In [ ]:
## Check CD4 IDs should not be amogn CD8 IDs
shared = list(set(CD4_ids) & set(CD8_ids))
shared

In [ ]:
obs_df = sdata["gex"].obs.copy()
obs_df[obs_df["final_ann_vote"] == "clone_not_neglibile_for_vote"][["clone_handle", "clone_size", "L2.5_ann"]].groupby("clone_size").size().reset_index()  #Not eligible for voting are these low expanded clones


In [ ]:
pd.crosstab(
    sdata["gex"].obs["L2.5_ann"], 
    sdata["gex"].obs["final_ann_vote"])

In [ ]:
#Option X: Do NOT correct DPs!
# sdata["gex"].obs["final_ann"] = np.where(~sdata["gex"].obs["L2.5_ann"].isin(["CD4 T cells", "CD8 T cells", "CD4/CD8 T cells"]), sdata["gex"].obs["L2.5_ann"],    ## non-T cells annotations are done first
#                                   np.where(sdata["gex"].obs["L2.5_ann"] == sdata["gex"].obs["final_ann_vote"], sdata["gex"].obs["L2.5_ann"],                     ## 1st priority - if labels are equal, change nothing. 
#                                   np.where(sdata["gex"].obs["L2.5_ann"] != sdata["gex"].obs["final_ann_vote"],                                                               ## 2nd priority - if labels are different do something...  
#                                   np.where(sdata["gex"].obs["L2.5_ann"] == "CD4/CD8 T cells", sdata["gex"].obs["L2.5_ann"],                                                   ## CD4/CD8 T cells are risky to correct here
#                                      np.where(sdata["gex"].obs["final_ann_vote"] == "clone_not_neglibile_for_vote", sdata["gex"].obs["L2.5_ann"],                            ## ---vote = clone_not_neglibile_for_vote - use original labels
#                                      np.where(sdata["gex"].obs["final_ann_vote"] == "tied", sdata["gex"].obs["L2.5_ann"],                                                    ## ---vote = tied - use original labels
#                                               sdata["gex"].obs["final_ann_vote"]))), "error")))                                                                               ## Otherwise, use the voted clone identity label as the final label.

# sdata["gex"].obs[["final_ann"]].value_counts()

#Option X: Correct with DPs!
sdata["gex"].obs["final_ann"] = np.where(~sdata["gex"].obs["L2.5_ann"].isin(["CD4 T cells", "CD8 T cells", "CD4/CD8 T cells"]), sdata["gex"].obs["L2.5_ann"],    ## non-T cells annotations are done first
                                  np.where(sdata["gex"].obs["L2.5_ann"] == sdata["gex"].obs["final_ann_vote"], sdata["gex"].obs["L2.5_ann"],                     ## 1st priority - if labels are equal, change nothing. 
                                  np.where(sdata["gex"].obs["L2.5_ann"] != sdata["gex"].obs["final_ann_vote"],                                                   ## 2ndd priority - if labels are different do something...  
                                                                                                                                                                         
                                     np.where(sdata["gex"].obs["final_ann_vote"] == "clone_not_neglibile_for_vote", sdata["gex"].obs["L2.5_ann"],                ## ---vote = clone_not_neglibile_for_vote - use original labels
                                     np.where(sdata["gex"].obs["final_ann_vote"] == "tied", sdata["gex"].obs["L2.5_ann"],                                        ## ---vote = tied - use original labels
                                              sdata["gex"].obs["final_ann_vote"])), "error")))                                                                  ## Otherwise, use the voted clone identity label as the final label.

sdata["gex"].obs[["final_ann"]].value_counts()

#### Look at intersecting clones again
Worked - no more intersecting clones

In [ ]:
# cd8_dat = sdata["gex"].obs[sdata["gex"].obs["final_ann_vote"] == "CD8 T cells"]["clone_handle"].drop_duplicates().tolist()
# cd4_dat = sdata["gex"].obs[sdata["gex"].obs["final_ann_vote"] == "CD4 T cells"]["clone_handle"].drop_duplicates().tolist()

# CD4_TCRs = set(cd4_dat)
# CD8_TCRs = set(cd8_dat)

# set1 = CD4_TCRs
# set2 = CD8_TCRs

# # Compute sizes for each section
# only_set1 = len(set1 - set2)
# only_set2 = len(set2 - set1)
# intersection = len(set1 & set2)

# total = only_set1 + only_set2 + intersection

# # Compute percentages
# pct1 = 100 * only_set1 / total
# pct2 = 100 * only_set2 / total
# pct_inter = 100 * intersection / total

# # Create a Venn diagram
# vennAll = venn2(subsets=(only_set1, only_set2, intersection), set_labels=('CD4 TCRs', 'CD8 TCRs'))

# # Modify labels to show percentages
# if vennAll.get_label_by_id('10'):
#     vennAll.get_label_by_id('10').set_text(f"{only_set1}\n({pct1:.1f}%)")
# if vennAll.get_label_by_id('01'):
#     vennAll.get_label_by_id('01').set_text(f"{only_set2}\n({pct2:.1f}%)")
# if vennAll.get_label_by_id('11'):
#     vennAll.get_label_by_id('11').set_text(f"{intersection}\n({pct_inter:.1f}%)")

# # Set colors to a colorblind-friendly palette
# vennAll.get_patch_by_id('10').set_color('#ff0000')  # left = Red
# vennAll.get_patch_by_id('01').set_color('#1f77b4')  # right = Blue
# vennAll.get_patch_by_id('11').set_color('#ff7f0e')  # overlap = orange

# # Adjust transparency for all patches
# for venn in [vennAll]:
#     for patch in venn.patches:
#         if patch:
#             patch.set_alpha(0.7)

# # Display the plot
# plt.title("Clonotype overlap, CD4 vs CD8") 
# plt.tight_layout()
# plt.show()

# #2% overlap when CD4/CD8 T cells are included
# #0.5% overlap when CD4 v CD8 are excluded

# L2.5_ann vs final_ann

In [ ]:
pd.crosstab(
    sdata["gex"].obs["L2.5_ann"], 
    sdata["gex"].obs["final_ann"])

## Consistency check
How much of each co-receptor du the different categories of cd4cd8_vote express?

In [ ]:
coreceptors = {
    "T-cell status": ["CD3E", "CD4", "CD8A", "CD8B"]
}

sc.pl.dotplot(sdata["gex"], coreceptors, groupby="final_ann", standard_scale="var", color_map='RdYlBu_r')


In [ ]:
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "cd4cd8_gex_assignment", lambda x: x != "negative")

# Visualize

In [ ]:
plt.rcParams["figure.figsize"] = (5, 5)

sc.pl.umap(
    sdata["gex"],
    color=["final_ann"],
    legend_fontweight="bold",
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=10,
    show=False
)

# plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/36_GlobalAnalysis_umap.pdf",
    dpi=400,
    
)
plt.show()

plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Select obs clusters to keep

In [ ]:
sdata["gex"].obs.columns

In [ ]:
cols_to_keep = ["scrublet_score", "n_genes_by_counts", "total_counts", "pct_counts_mt", "outlier", 'antigen.species', 'antigen.gene',  'antigen.epitope',
               "chain_pairing", "pool", "sample", "batch", 'CD45RA_adt', 'CD62L_adt', 'CD95_adt', "CD4_adt", "CD8_adt", 
               "Compartment", "donor",  "clone_id", "clone_handle", "cd4cd8_cs_assignment", "cd4cd8_combined_assignment", 
               "leiden_4.5", "pct_counts_ribo", "L1_ann", "final_ann"]

sdata["gex"].obs = sdata["gex"].obs[cols_to_keep].copy()
sdata.update()

# Export

In [ ]:
sdata.write(filename="../../04_data_objects/06_preprocessed_v2/2026_6_main_dataset_clustered_annotated.h5mu")

# Graveyard

# Basic annotation - multimer specificity
Discussion for creating a dictionary from columns: 
- https://stackoverflow.com/questions/17426292/how-to-create-a-dictionary-of-two-pandas-dataframe-columns
- https://stackoverflow.com/questions/71753000/modify-a-dataframe-column-with-a-dictionary #For the map solution

In [ ]:
mlt = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/clone_x_multimer_assignments.xlsx").dropna()

## The clone_handles in obs contains .0. Remove these first: 
# Normalize obs clone_handle: 'Pt1-2.0' -> 'Pt1-2', 'Pt1-nan' stays 'Pt1-nan'
sdata["gex"].obs['clone_handle'] = (
    sdata["gex"].obs['clone_handle']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
)

# Build and apply dictionaries using the clean key
for col, new_col in [('handle', 'multimer'), ('Virus', 'Virus'), ('HLA', 'HLA')]:
    mlt_dic = mlt.set_index('clone_handle').to_dict()[col]
    sdata["gex"].obs[new_col] = (
        sdata["gex"].obs['clone_handle']
        .map(mlt_dic)
    )

# Verify
print(sdata["gex"].obs[['clone_handle', 'multimer', 'Virus', 'HLA']].sort_values("HLA").head(2))

In [ ]:
# ## Modify the pallette
# virus_palette = {
#     'CMV':        '#1f77b4',  # blue
#     'EBV':        '#ff7f0e',  # orange
#     'FLU-A':      '#2ca02c',  # green
#     'HAdV-C':     '#d62728',  # red
#     'HHV-1':      '#9467bd',  # purple
#     'HHV-6B':     '#8c564b',  # brown
#     'SARS-COV-2': '#e377c2',  # pink
#     'VZV':        '#bcbd22',  # yellow-green
#     'unknown':    '#d3d3d3',  # grey
# }


In [ ]:
plt.rcParams["figure.figsize"] = (7, 6)
sc.pl.umap(sdata["gex"], color=["Virus"],  legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2,  size=40)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Basic annotation - EBV reads

In [ ]:
## Load the .gtv file
ebv_df = pd.read_csv(
    "../../03_annotation/03_reference_genomes/chrEBV.tsv",
    sep="\t"
)

#Rename columns
ebv_df.columns = [
    "seqname",     # chrEBV
    "source",      # JohannsenLab
    "feature",     # gene / exon / CDS
    "start",       # genomic start
    "end",         # genomic end
    "score",       # usually "."
    "strand",      # + / -
    "frame",       # phase
    "attributes"   # gene_id "..."; gene_name "...";
]

# Parse or same the gene_id and gene_name
ebv_df["gene_id"] = ebv_df["attributes"].str.extract(r'gene_id "([^"]+)"')
ebv_df["gene_name"] = ebv_df["attributes"].str.extract(r'gene_name "([^"]+)"')

# Get unique names
ebv_df[["gene_id", "gene_name"]].drop_duplicates()
ebv_df[ebv_df["gene_id"] != ebv_df["gene_name"]] #They're always the same name

# Get a list of gene names
ebv_genes = ebv_df["gene_id"].drop_duplicates().to_list()

## annotate the .obs where cells have transcripts
gex = sdata["gex"]

# Intersect EBV genes with genes present in the dataset
ebv_genes_present = gex.var_names.intersection(ebv_genes)

# Sum EBV expression per cell (sparse- and dense-safe)
X_ebv = gex[:, ebv_genes_present].X
ebv_counts = np.asarray(X_ebv.sum(axis=1)).ravel()

# Annotate cells with any EBV transcript
gex.obs["EBV_transcripts"] = ebv_counts

In [ ]:
plt.rcParams["figure.figsize"] = (7, 6)
sc.pl.umap(sdata["gex"], color=["EBV_transcripts"], legend_fontweight='bold', frameon= False, legend_fontoutline=2, cmap="RdBu_r", size=40)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Explore

### Basic annotation - summarized

In [ ]:
## Simple clusters to identify T cell versus myeloid cell
marker_genes = {
    "T cells": ["CD3E", "TRBC2", "THEMIS", "TRAC", "TRBC1"],
    "Antigen-presentation": ["HLA-DRA", "HLA-DRB1", "CD74"],
    "Complement genes": ["C1QA", "C1QB", "C1QC"],
    "Monocyte genes": ["FCN1"], 
    "CD14+ Mono":["CD14"],
    "CD16+ Mono":["CD68", "FCGR3A"],
    "BAM": ["CYBB", "F13A1", "MS4A7","LYVE1", "CD163", "MRC1"],
    "BAM-associated": ["APOE", "TGFBI", "APOC1", "ITGAX"],
    "Microglia": ["SPP1", "TREM2", "CCL2", "CLEC7A", "ITGAX", "CX3CR1", "P2RY12", "TMEM119"],
    "mDCs": ["FLT3", "CLEC9A", "CD1C", "CD1E", "FCER1A"],
    "B cells": ["CD19", "MS4A1", "CD79A", "IGHG1","IGKC", "JCHAIN"],
    "pDCs": ["LILRA4", "IRF8"],
    "CITEseq": ["CD4_adt", "CD8_adt"],
    "CD4 vs CD8": ["CD4", "CD8A"],
    "MAIT" : ["TRAV1-2", "SLC4A10"],
    "GD T cells": ["TRGV9", "TRDV1"],
    "Effector": ["CCL5", "GZMK", "GZMB", "GNLY", "NKG7", "LAG3", "TIGIT"],
    "IFN-sig": ["IFI44", "IFI44L", "ISG15", "OAS2"], 
    "T naive": ["LEF1", "CCR7", "TCF7", "SELL"],
    "T reg": ["FOXP3", "IL2RA", "CTLA4"],
    "NK cells": ["NCAM1", "NCR1", "KIR2DL4", 
                 "KLRB1", #CD161
                 "KLRC2", #NKG2A specifically
                 "KLRK1", #NKG2D specifically
                 "KLRC1", #NKG2A,D,C
                 "KLRD1"], #CD94
    "Ribosomal_genes": ["RPLP1", "EEF1A1", "RPL28"]
    # "AXL+SIGLEC6+ DCs":["PPP1R14A", "AXL", "SIGLEC6"],
}

In [ ]:
sc.pl.dotplot(sdata["gex"], marker_genes, groupby="L1_ann", standard_scale="var")